# Random Forest 16 kHz — Standalone Inference & Evaluation

## Goal

Notebook này chạy lại **frozen canonical Random Forest** mà không import package
`drone_sound` và không train/tune lại model.

```text
WAV -> mono -> resample_poly 16 kHz toàn file -> window 1.0 s / stride 0.5 s
    -> 100 spectral_mfcc_v1 features -> frozen train scaler
    -> Random Forest P(drone) -> frozen threshold_f2
```

Notebook hỗ trợ hai đường kiểm chứng:

1. Tái tính nhanh kết quả từ canonical feature bundles.
2. Rebuild Private real-only trực tiếp từ 11 WAV gốc để kiểm tra end-to-end.

Lưu ý: Private real-only chỉ có 1 positive recording và 10 negative recordings;
các window overlap nên đây là external stress test, không phải deployment estimate.


## Setup

Python 3.13 và các package cần thiết:

```bash
python -m pip install numpy scipy joblib scikit-learn
```

Chỉ load `.joblib` từ nguồn tin cậy. Notebook mặc định tìm artifact theo đường dẫn
repo-relative; nếu copy ra ngoài repo, sửa các path trong cell **Configuration**.


In [1]:
#!/usr/bin/env python3
from __future__ import annotations

import argparse
import csv
import json
import math
import struct
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Mapping, Sequence

import joblib
import numpy as np
from scipy import fft as scipy_fft
from scipy.signal import resample_poly


SCRIPT_DIR = Path.cwd()
POSITIVE_LABEL = "drone"
NEGATIVE_LABEL = "no_drone"
TARGET_SAMPLE_RATE_HZ = 16_000
PREPROCESSING_PROFILE = "mono_resample_poly_16khz_v1"
FEATURE_SET = "spectral_mfcc_v1"
WINDOW_SECONDS = 1.0
STRIDE_SECONDS = 0.5
EPSILON = 1e-12
MFCC_COEFFICIENT_COUNT = 20
MEL_FILTER_COUNT = 40
LOW_BAND_HZ = (20.0, 500.0)
MID_BAND_HZ = (500.0, 2_000.0)
WAVE_FORMAT_PCM = 1
WAVE_FORMAT_IEEE_FLOAT = 3
WAVE_FORMAT_EXTENSIBLE = 65534

V1_BASIC_FEATURE_NAMES = (
    "mean",
    "mean_abs",
    "rms",
    "peak_abs",
    "zero_crossing_rate",
)

SPECTRAL_FEATURE_NAMES = (
    "spectral_centroid_hz",
    "spectral_bandwidth_hz",
    "spectral_rolloff_85_hz",
    "spectral_flatness",
    "spectral_contrast_db",
    "dominant_frequency_hz",
    "low_band_energy",
    "mid_band_energy",
    "high_band_energy",
    "low_total_ratio",
    "mid_total_ratio",
    "high_total_ratio",
    "low_mid_ratio",
    "low_high_ratio",
    "mid_high_ratio",
)

MFCC_FEATURE_NAMES = (
    tuple(f"mfcc_{index:02d}_mean" for index in range(1, MFCC_COEFFICIENT_COUNT + 1))
    + tuple(f"mfcc_{index:02d}_std" for index in range(1, MFCC_COEFFICIENT_COUNT + 1))
    + tuple(f"delta_mfcc_{index:02d}_mean" for index in range(1, MFCC_COEFFICIENT_COUNT + 1))
    + tuple(f"delta_mfcc_{index:02d}_std" for index in range(1, MFCC_COEFFICIENT_COUNT + 1))
)

FEATURE_NAMES = V1_BASIC_FEATURE_NAMES + SPECTRAL_FEATURE_NAMES + MFCC_FEATURE_NAMES


@dataclass(frozen=True)
class WavMetadata:
    path: Path
    format_tag: int
    decoded_format_tag: int
    channels: int
    sample_rate_hz: int
    sample_width_bytes: int
    bits_per_sample: int
    block_align: int
    frames: int
    duration_sec: float
    data_offset: int
    data_size_bytes: int



def canonicalize_audio(samples: Any, source_sample_rate_hz: int) -> np.ndarray:
    """Downmixing happens during WAV decoding; resample the complete file once."""
    if source_sample_rate_hz <= 0:
        raise ValueError("source sample rate must be positive")
    array = np.asarray(samples, dtype=np.float64).reshape(-1)
    if not np.all(np.isfinite(array)):
        raise ValueError("audio contains non-finite samples")
    if source_sample_rate_hz == TARGET_SAMPLE_RATE_HZ:
        return array.copy()
    divisor = math.gcd(source_sample_rate_hz, TARGET_SAMPLE_RATE_HZ)
    up = TARGET_SAMPLE_RATE_HZ // divisor
    down = source_sample_rate_hz // divisor
    output = np.asarray(resample_poly(array, up, down), dtype=np.float64)
    expected = int(round(len(array) * TARGET_SAMPLE_RATE_HZ / source_sample_rate_hz))
    if len(output) > expected:
        output = output[:expected]
    elif len(output) < expected:
        output = np.pad(output, (0, expected - len(output)))
    return np.ascontiguousarray(output, dtype=np.float64)


def _discover_wavs(dataset_root: Path, include_calibration: bool) -> list[Path]:
    files = sorted(path for path in dataset_root.rglob("*.wav") if path.is_file())
    if include_calibration:
        return files
    return [path for path in files if not path.name.lower().startswith("calib_")]


def _default_artifact_path(repo_relative: str, local_name: str) -> Path:
    repo_path = Path(repo_relative)
    if repo_path.exists():
        return repo_path
    return SCRIPT_DIR / local_name


def _extract_dataset_features(
    wav_files: Sequence[Path],
    *,
    progress_every: int,
) -> tuple[list[dict[str, Any]], np.ndarray, dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    features: list[tuple[float, ...]] = []
    started_at = time.monotonic()
    non_finite_replacements = 0
    skipped_files: list[dict[str, str]] = []
    total_files = len(wav_files)

    for file_index, path in enumerate(wav_files, start=1):
        try:
            samples, metadata = read_wav_mono_array(path)
        except Exception as error:
            skipped_files.append({"path": str(path), "error": str(error)})
            print(f"feature progress: skipped file={path} error={error}", file=sys.stderr, flush=True)
            continue
        samples = canonicalize_audio(samples, metadata.sample_rate_hz)
        window_size = int(round(WINDOW_SECONDS * TARGET_SAMPLE_RATE_HZ))
        hop_size = int(round(STRIDE_SECONDS * TARGET_SAMPLE_RATE_HZ))
        if window_size <= 0 or hop_size <= 0 or len(samples) < window_size:
            skipped_files.append({"path": str(path), "error": "shorter than one complete window"})
            continue
        starts = range(0, len(samples) - window_size + 1, hop_size)
        file_window_count = 0
        for start_index in starts:
            end_index = start_index + window_size
            vector, replacements = extract_feature_vector(
                samples[start_index:end_index],
                TARGET_SAMPLE_RATE_HZ,
            )
            non_finite_replacements += replacements
            start_sec = start_index / TARGET_SAMPLE_RATE_HZ
            end_sec = end_index / TARGET_SAMPLE_RATE_HZ
            rows.append(
                {
                    "file": str(path),
                    "file_name": path.name,
                    "sample_rate_hz": TARGET_SAMPLE_RATE_HZ,
                    "source_sample_rate_hz": metadata.sample_rate_hz,
                    "processed_sample_rate_hz": TARGET_SAMPLE_RATE_HZ,
                    "window_start_sec": start_sec,
                    "window_end_sec": end_sec,
                }
            )
            features.append(vector)
            file_window_count += 1
            if progress_every > 0 and len(rows) % progress_every == 0:
                _print_feature_progress(
                    len(rows),
                    file_index,
                    total_files,
                    started_at,
                    current_file=path,
                )
        print(
            "feature progress: "
            f"file={file_index}/{total_files} windows={file_window_count} "
            f"source_rate={metadata.sample_rate_hz} processed_rate={TARGET_SAMPLE_RATE_HZ} "
            f"path={path.name}",
            flush=True,
        )

    if not features:
        raise RuntimeError("no feature rows extracted")
    _print_feature_progress(len(rows), total_files, total_files, started_at, current_file=wav_files[-1])
    return (
        rows,
        np.asarray(features, dtype=np.float64),
        {
            "non_finite_replacements": non_finite_replacements,
            "skipped_files": skipped_files,
            "elapsed_seconds": time.monotonic() - started_at,
        },
    )


def _print_feature_progress(
    processed_windows: int,
    file_index: int,
    total_files: int,
    started_at: float,
    *,
    current_file: Path,
) -> None:
    elapsed = max(time.monotonic() - started_at, 0.0)
    rate = processed_windows / elapsed if elapsed > 0 else 0.0
    print(
        "feature progress: "
        f"windows={processed_windows} files={file_index}/{total_files} "
        f"rate={rate:.1f} windows/s elapsed={_format_duration(elapsed)} "
        f"current_file={current_file.name}",
        flush=True,
    )


def extract_feature_vector(samples: Any, sample_rate_hz: int) -> tuple[tuple[float, ...], int]:
    array = np.asarray(samples, dtype=np.float64)
    if array.ndim != 1:
        array = array.reshape(-1)
    if array.size == 0:
        raise ValueError("cannot extract features from an empty audio window")
    values = _v1_basic_features(array) + _spectral_features(array, sample_rate_hz) + _mfcc_features(array, sample_rate_hz)
    return _finite_tuple(values)


def _v1_basic_features(samples: np.ndarray) -> tuple[float, ...]:
    abs_samples = np.abs(samples)
    signs = samples >= 0
    crossings = np.count_nonzero(signs[1:] != signs[:-1]) if len(samples) > 1 else 0
    zero_crossing_rate = crossings / (len(samples) - 1) if len(samples) > 1 else 0.0
    return (
        float(np.mean(samples)),
        float(np.mean(abs_samples)),
        float(np.sqrt(np.mean(samples * samples))),
        float(np.max(abs_samples)),
        float(zero_crossing_rate),
    )


def _spectral_features(samples: np.ndarray, sample_rate_hz: int) -> tuple[float, ...]:
    windowed = samples * np.hanning(len(samples))
    magnitudes = np.abs(np.fft.rfft(windowed))
    power = magnitudes * magnitudes
    freqs = np.fft.rfftfreq(len(samples), d=1.0 / sample_rate_hz)
    total_power = float(np.sum(power))
    if total_power <= EPSILON:
        return (0.0,) * len(SPECTRAL_FEATURE_NAMES)

    centroid = float(np.sum(freqs * power) / total_power)
    bandwidth = float(np.sqrt(np.sum(((freqs - centroid) ** 2) * power) / total_power))
    cumulative = np.cumsum(power)
    rolloff_index = int(np.searchsorted(cumulative, 0.85 * total_power, side="left"))
    rolloff_index = min(rolloff_index, len(freqs) - 1)
    rolloff = float(freqs[rolloff_index])
    flatness = float(np.exp(np.mean(np.log(power + EPSILON))) / (np.mean(power) + EPSILON))
    positive = magnitudes[magnitudes > EPSILON]
    if positive.size:
        low_percentile = float(np.percentile(positive, 5))
        high_percentile = float(np.percentile(positive, 95))
        spectral_contrast = 20.0 * math.log10((high_percentile + EPSILON) / (low_percentile + EPSILON))
    else:
        spectral_contrast = 0.0
    dominant_frequency = float(freqs[int(np.argmax(power))])

    low_energy = _band_energy(freqs, power, LOW_BAND_HZ[0], LOW_BAND_HZ[1])
    mid_energy = _band_energy(freqs, power, MID_BAND_HZ[0], MID_BAND_HZ[1])
    high_energy = _band_energy(freqs, power, MID_BAND_HZ[1], sample_rate_hz / 2.0)
    return (
        centroid,
        bandwidth,
        rolloff,
        flatness,
        spectral_contrast,
        dominant_frequency,
        low_energy,
        mid_energy,
        high_energy,
        low_energy / (total_power + EPSILON),
        mid_energy / (total_power + EPSILON),
        high_energy / (total_power + EPSILON),
        low_energy / (mid_energy + EPSILON),
        low_energy / (high_energy + EPSILON),
        mid_energy / (high_energy + EPSILON),
    )


def _mfcc_features(samples: np.ndarray, sample_rate_hz: int) -> tuple[float, ...]:
    frames = _framed_audio(samples, sample_rate_hz)
    frame_length = frames.shape[1]
    n_fft = _next_power_of_two(frame_length)
    window = np.hanning(frame_length)
    spectrum = np.fft.rfft(frames * window, n=n_fft, axis=1)
    power = (np.abs(spectrum) ** 2) / n_fft
    filters = _mel_filterbank(sample_rate_hz, n_fft, MEL_FILTER_COUNT)
    mel_energy = np.maximum(power @ filters.T, EPSILON)
    log_mel = np.log(mel_energy)
    coefficients = scipy_fft.dct(log_mel, type=2, norm="ortho", axis=1)[:, :MFCC_COEFFICIENT_COUNT]
    if coefficients.shape[0] > 1:
        deltas = np.diff(coefficients, axis=0)
    else:
        deltas = np.zeros_like(coefficients)
    return tuple(
        [float(value) for value in np.mean(coefficients, axis=0)]
        + [float(value) for value in np.std(coefficients, axis=0)]
        + [float(value) for value in np.mean(deltas, axis=0)]
        + [float(value) for value in np.std(deltas, axis=0)]
    )


def _framed_audio(samples: np.ndarray, sample_rate_hz: int) -> np.ndarray:
    frame_length = max(1, int(round(0.025 * sample_rate_hz)))
    hop_length = max(1, int(round(0.010 * sample_rate_hz)))
    if len(samples) < frame_length:
        samples = np.pad(samples, (0, frame_length - len(samples)))
    frame_starts = range(0, len(samples) - frame_length + 1, hop_length)
    frames = [samples[start : start + frame_length] for start in frame_starts]
    if not frames:
        frames = [np.pad(samples, (0, frame_length - len(samples)))]
    return np.vstack(frames)


def _mel_filterbank(sample_rate_hz: int, n_fft: int, mel_count: int) -> np.ndarray:
    min_mel = _hz_to_mel(0.0)
    max_mel = _hz_to_mel(sample_rate_hz / 2.0)
    mel_points = np.linspace(min_mel, max_mel, mel_count + 2)
    hz_points = _mel_to_hz(mel_points)
    bins = np.floor((n_fft + 1) * hz_points / sample_rate_hz).astype(int)
    bins = np.clip(bins, 0, n_fft // 2)
    filters = np.zeros((mel_count, n_fft // 2 + 1), dtype=np.float64)
    for index in range(1, mel_count + 1):
        left, center, right = bins[index - 1], bins[index], bins[index + 1]
        if center == left:
            center = min(center + 1, n_fft // 2)
        if right == center:
            right = min(right + 1, n_fft // 2)
        for bin_index in range(left, center):
            filters[index - 1, bin_index] = (bin_index - left) / max(center - left, 1)
        for bin_index in range(center, right):
            filters[index - 1, bin_index] = (right - bin_index) / max(right - center, 1)
    return filters


def _hz_to_mel(value: Any) -> Any:
    return 2595.0 * np.log10(1.0 + np.asarray(value) / 700.0)


def _mel_to_hz(value: Any) -> Any:
    return 700.0 * (10.0 ** (np.asarray(value) / 2595.0) - 1.0)


def _band_energy(freqs: np.ndarray, power: np.ndarray, low_hz: float, high_hz: float) -> float:
    mask = (freqs >= low_hz) & (freqs < high_hz)
    return float(np.sum(power[mask]))


def _finite_tuple(values: tuple[float, ...]) -> tuple[tuple[float, ...], int]:
    sanitized = []
    replacements = 0
    for value in values:
        if math.isfinite(value):
            sanitized.append(float(value))
        else:
            sanitized.append(0.0)
            replacements += 1
    return tuple(sanitized), replacements


def _next_power_of_two(value: int) -> int:
    return 1 << (value - 1).bit_length()


def _load_scaler(path: Path) -> dict[str, Any]:
    data = json.loads(path.read_text(encoding="utf-8"))
    if data.get("feature_set") != FEATURE_SET:
        raise ValueError(f"scaler feature_set must be {FEATURE_SET}")
    if tuple(data.get("feature_names", [])) != FEATURE_NAMES:
        raise ValueError("scaler feature_names do not match standalone extractor")
    if len(data.get("mean", [])) != len(FEATURE_NAMES) or len(data.get("std", [])) != len(FEATURE_NAMES):
        raise ValueError("scaler mean/std dimensions do not match feature count")
    return data


def _apply_scaler(features: np.ndarray, scaler: Mapping[str, Any]) -> np.ndarray:
    mean = np.asarray(scaler["mean"], dtype=np.float64)
    std = np.asarray(scaler["std"], dtype=np.float64)
    if features.ndim != 2:
        raise ValueError("features must be a 2D matrix")
    if features.shape[1] != len(mean) or len(mean) != len(std):
        raise ValueError("scaler dimensions do not match features")
    return (features - mean) / std


def _load_model(path: Path) -> dict[str, Any]:
    model = joblib.load(path)
    if not isinstance(model, dict):
        raise ValueError(f"{path}: model artifact must be a dict package")
    return model


def _validate_model_package(name: str, model: Mapping[str, Any]) -> None:
    required = {
        "feature_set",
        "feature_names",
        "labels",
        "threshold",
        "threshold_name",
        "estimator",
    }
    missing = sorted(required - set(model))
    if missing:
        raise ValueError(f"{name}: model package missing keys: {', '.join(missing)}")
    if model["feature_set"] != FEATURE_SET:
        raise ValueError(f"{name}: feature_set must be {FEATURE_SET}")
    if tuple(model["feature_names"]) != FEATURE_NAMES:
        raise ValueError(f"{name}: feature_names do not match standalone extractor")
    labels = model["labels"]
    if labels.get("positive") != POSITIVE_LABEL:
        raise ValueError(f"{name}: positive label must be {POSITIVE_LABEL}")


def _score_model(
    model_name: str,
    estimator: Any,
    features: np.ndarray,
    *,
    batch_size: int,
) -> np.ndarray:
    scores: list[float] = []
    started_at = time.monotonic()
    total = len(features)
    for start in range(0, total, batch_size):
        batch = features[start : start + batch_size]
        scores.extend(_score_estimator(estimator, batch))
        processed = min(start + batch_size, total)
        elapsed = max(time.monotonic() - started_at, 0.0)
        rate = processed / elapsed if elapsed > 0 else 0.0
        eta = (total - processed) / rate if rate > 0 else 0.0
        print(
            "score progress: "
            f"model={model_name} windows={processed}/{total} "
            f"rate={rate:.1f} windows/s elapsed={_format_duration(elapsed)} "
            f"eta={_format_duration(eta)}",
            flush=True,
        )
    return np.asarray(scores, dtype=np.float64)


def _score_estimator(estimator: Any, features: np.ndarray) -> list[float]:
    if hasattr(estimator, "predict_proba"):
        probabilities = estimator.predict_proba(features)
        classes = list(getattr(estimator, "classes_", []))
        if POSITIVE_LABEL not in classes:
            raise ValueError("estimator classes do not include the positive label")
        positive_index = classes.index(POSITIVE_LABEL)
        return [float(value) for value in probabilities[:, positive_index]]
    if hasattr(estimator, "decision_function"):
        decision = np.asarray(estimator.decision_function(features), dtype=np.float64)
        if decision.ndim == 2:
            classes = list(getattr(estimator, "classes_", []))
            positive_index = classes.index(POSITIVE_LABEL) if POSITIVE_LABEL in classes else 1
            decision = decision[:, positive_index]
        return [float(1.0 / (1.0 + math.exp(-value))) for value in decision]
    raise ValueError("estimator does not support predict_proba or decision_function")


def _summarize_scores(
    rows: Sequence[Mapping[str, Any]],
    scores: np.ndarray,
    detections: np.ndarray,
    threshold: float,
) -> dict[str, Any]:
    file_count = len({str(row["file"]) for row in rows})
    detected_files = {
        str(row["file"])
        for row, detected in zip(rows, detections, strict=True)
        if bool(detected)
    }
    return {
        "threshold": threshold,
        "windows": int(len(rows)),
        "detected_windows": int(np.count_nonzero(detections)),
        "window_detection_rate": float(np.mean(detections)) if len(detections) else 0.0,
        "files": file_count,
        "files_with_detection": len(detected_files),
        "file_detection_rate": len(detected_files) / file_count if file_count else 0.0,
        "score_mean": float(np.mean(scores)) if len(scores) else None,
        "score_median": float(np.median(scores)) if len(scores) else None,
        "score_p05": float(np.quantile(scores, 0.05)) if len(scores) else None,
        "score_p95": float(np.quantile(scores, 0.95)) if len(scores) else None,
        "score_max": float(np.max(scores)) if len(scores) else None,
    }


def _write_window_csv(
    path: Path,
    rows: Sequence[Mapping[str, Any]],
    scores: np.ndarray,
    detections: np.ndarray,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(
            file,
            fieldnames=[
                "file",
                "file_name",
                "sample_rate_hz",
                "window_start_sec",
                "window_end_sec",
                "score",
                "detected",
            ],
        )
        writer.writeheader()
        for row, score, detected in zip(rows, scores, detections, strict=True):
            writer.writerow(
                {
                    "file": row["file"],
                    "file_name": row["file_name"],
                    "sample_rate_hz": row["sample_rate_hz"],
                    "window_start_sec": f"{float(row['window_start_sec']):.3f}",
                    "window_end_sec": f"{float(row['window_end_sec']):.3f}",
                    "score": f"{float(score):.9f}",
                    "detected": int(bool(detected)),
                }
            )


def read_wav_metadata(path: Path) -> WavMetadata:
    with path.open("rb") as file:
        if file.read(4) != b"RIFF":
            raise ValueError(f"{path}: not a RIFF file")
        file.seek(8)
        if file.read(4) != b"WAVE":
            raise ValueError(f"{path}: not a WAVE file")

        fmt_payload: bytes | None = None
        data_offset: int | None = None
        data_size: int | None = None

        while True:
            header = file.read(8)
            if not header:
                break
            if len(header) != 8:
                raise ValueError(f"{path}: truncated chunk header")
            chunk_id, chunk_size = struct.unpack("<4sI", header)
            chunk_data_offset = file.tell()
            if chunk_id == b"fmt ":
                fmt_payload = file.read(chunk_size)
            elif chunk_id == b"data":
                data_offset = chunk_data_offset
                data_size = chunk_size
                file.seek(chunk_size, 1)
            else:
                file.seek(chunk_size, 1)
            if chunk_size % 2:
                file.seek(1, 1)

    if fmt_payload is None:
        raise ValueError(f"{path}: missing fmt chunk")
    if data_offset is None or data_size is None:
        raise ValueError(f"{path}: missing data chunk")
    if len(fmt_payload) < 16:
        raise ValueError(f"{path}: invalid fmt chunk")

    format_tag, channels, sample_rate_hz, _, block_align, bits_per_sample = struct.unpack(
        "<HHIIHH",
        fmt_payload[:16],
    )
    decoded_format_tag = _decoded_format_tag(format_tag, fmt_payload, path)
    if decoded_format_tag not in {WAVE_FORMAT_PCM, WAVE_FORMAT_IEEE_FLOAT}:
        raise ValueError(f"{path}: unsupported WAV format tag {format_tag}")
    if channels < 1:
        raise ValueError(f"{path}: WAV must have at least one channel")
    if sample_rate_hz <= 0:
        raise ValueError(f"{path}: invalid sample rate")
    if bits_per_sample % 8 != 0:
        raise ValueError(f"{path}: unsupported bit depth {bits_per_sample}")
    sample_width_bytes = bits_per_sample // 8
    frames = data_size // block_align
    return WavMetadata(
        path=path,
        format_tag=format_tag,
        decoded_format_tag=decoded_format_tag,
        channels=channels,
        sample_rate_hz=sample_rate_hz,
        sample_width_bytes=sample_width_bytes,
        bits_per_sample=bits_per_sample,
        block_align=block_align,
        frames=frames,
        duration_sec=frames / sample_rate_hz,
        data_offset=data_offset,
        data_size_bytes=data_size,
    )


def read_wav_mono_array(path: Path) -> tuple[np.ndarray, WavMetadata]:
    metadata = read_wav_metadata(path)
    with metadata.path.open("rb") as file:
        file.seek(metadata.data_offset)
        data = file.read(metadata.data_size_bytes)
    if len(data) != metadata.data_size_bytes:
        raise ValueError(f"{path}: truncated audio data")

    if metadata.decoded_format_tag == WAVE_FORMAT_IEEE_FLOAT:
        if metadata.sample_width_bytes == 4:
            array = np.frombuffer(data, dtype="<f4")
        elif metadata.sample_width_bytes == 8:
            array = np.frombuffer(data, dtype="<f8")
        else:
            raise ValueError(f"{metadata.path}: unsupported IEEE float width {metadata.sample_width_bytes}")
        return _mono_array(array, metadata.channels), metadata

    if metadata.sample_width_bytes == 1:
        array = (np.frombuffer(data, dtype=np.uint8).astype(np.float64) - 128.0) / 128.0
    elif metadata.sample_width_bytes == 2:
        array = np.frombuffer(data, dtype="<i2").astype(np.float64) / float(2**15)
    elif metadata.sample_width_bytes == 3:
        return np.asarray(_decode_frames(data, metadata), dtype=np.float64), metadata
    elif metadata.sample_width_bytes == 4:
        array = np.frombuffer(data, dtype="<i4").astype(np.float64) / float(2**31)
    else:
        raise ValueError(f"{metadata.path}: unsupported PCM width {metadata.sample_width_bytes}")
    return _mono_array(array, metadata.channels), metadata


def _mono_array(array: np.ndarray, channels: int) -> np.ndarray:
    if channels == 1:
        return array.astype(np.float64, copy=False)
    return array.reshape(-1, channels).mean(axis=1)


def _decoded_format_tag(format_tag: int, fmt_payload: bytes, path: Path) -> int:
    if format_tag != WAVE_FORMAT_EXTENSIBLE:
        return format_tag
    if len(fmt_payload) < 40:
        raise ValueError(f"{path}: invalid extensible fmt chunk")
    subformat = fmt_payload[24:40]
    return int.from_bytes(subformat[:2], byteorder="little")


def _decode_frames(data: bytes, metadata: WavMetadata) -> list[float]:
    samples: list[float] = []
    for frame_offset in range(0, len(data), metadata.block_align):
        channel_values = []
        for channel_index in range(metadata.channels):
            sample_offset = frame_offset + (channel_index * metadata.sample_width_bytes)
            raw = data[sample_offset : sample_offset + metadata.sample_width_bytes]
            if len(raw) != metadata.sample_width_bytes:
                continue
            channel_values.append(_decode_sample(raw, metadata))
        if channel_values:
            samples.append(sum(channel_values) / len(channel_values))
    return samples


def _decode_sample(raw: bytes, metadata: WavMetadata) -> float:
    if metadata.decoded_format_tag == WAVE_FORMAT_IEEE_FLOAT:
        if metadata.sample_width_bytes == 4:
            return float(struct.unpack("<f", raw)[0])
        if metadata.sample_width_bytes == 8:
            return float(struct.unpack("<d", raw)[0])
        raise ValueError(f"{metadata.path}: unsupported IEEE float width {metadata.sample_width_bytes}")
    if metadata.sample_width_bytes == 1:
        return (raw[0] - 128) / 128.0
    if metadata.sample_width_bytes == 2:
        return int.from_bytes(raw, byteorder="little", signed=True) / float(2**15)
    if metadata.sample_width_bytes == 3:
        sign_byte = b"\xff" if raw[2] & 0x80 else b"\x00"
        value = int.from_bytes(raw + sign_byte, byteorder="little", signed=True)
        return value / float(2**23)
    if metadata.sample_width_bytes == 4:
        return int.from_bytes(raw, byteorder="little", signed=True) / float(2**31)
    raise ValueError(f"{metadata.path}: unsupported PCM width {metadata.sample_width_bytes}")


def _require_file(path: Path, label: str) -> None:
    if not path.is_file():
        raise SystemExit(f"{label} does not exist or is not a file: {path}")


def _require_dir(path: Path, label: str) -> None:
    if not path.is_dir():
        raise SystemExit(f"{label} does not exist or is not a directory: {path}")


def _format_duration(seconds: float | None) -> str:
    if seconds is None:
        return "n/a"
    seconds = max(float(seconds), 0.0)
    if seconds < 60:
        return f"{seconds:.1f}s"
    minutes, rem = divmod(int(round(seconds)), 60)
    hours, minutes = divmod(minutes, 60)
    if hours:
        return f"{hours}h{minutes:02d}m{rem:02d}s"
    return f"{minutes}m{rem:02d}s"


def _jsonable(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(key): _jsonable(item) for key, item in value.items()}
    if isinstance(value, list):
        return [_jsonable(item) for item in value]
    if isinstance(value, tuple):
        return [_jsonable(item) for item in value]
    if isinstance(value, np.generic):
        return value.item()
    return value


from types import SimpleNamespace

infer = SimpleNamespace(
    FEATURE_SET=FEATURE_SET,
    FEATURE_NAMES=FEATURE_NAMES,
    _discover_wavs=_discover_wavs,
    read_wav_metadata=read_wav_metadata,
    _load_scaler=_load_scaler,
    _load_model=_load_model,
    _validate_model_package=_validate_model_package,
    _extract_dataset_features=_extract_dataset_features,
    _apply_scaler=_apply_scaler,
    _score_model=_score_model,
    _summarize_scores=_summarize_scores,
    _write_window_csv=_write_window_csv,
    canonicalize_audio=canonicalize_audio,
    _jsonable=_jsonable,
)

def first_existing(*paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    return Path(paths[0])

print("Loaded self-contained inference helpers from this notebook")
print(f"Feature set: {infer.FEATURE_SET}")
print(f"Feature count: {len(infer.FEATURE_NAMES)}")


Loaded self-contained inference helpers from this notebook
Feature set: spectral_mfcc_v1
Feature count: 100


## Configuration

- `RUN_PUBLIC_BUNDLE_EVALUATION`: replay Public test/Professor khi public bundle có sẵn.
- `RUN_PRIVATE_BUNDLE_EVALUATION`: replay riêng Private real-only từ frozen bundle.
- `RUN_RAW_PRIVATE_REBUILD`: đọc lại 11 WAV private, preprocess và extract feature
  từ đầu trước khi inference.
- `WRITE_OUTPUT_FILES`: bật nếu muốn ghi CSV/JSON mới; mặc định tắt để notebook
  không thay đổi artifact hiện có.


In [2]:
def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "artifacts" / "v2_16khz").is_dir():
            return candidate
    return start


def first_existing(*paths: Path) -> Path:
    for path in paths:
        if path.exists():
            return path
    return paths[0]


REPO_ROOT = find_repo_root(Path.cwd())
LOCAL_ARTIFACT_DIR = Path.cwd()

MODEL_PATH = first_existing(
    REPO_ROOT / "artifacts/v2_16khz/models/v2a/models/random_forest.joblib",
    LOCAL_ARTIFACT_DIR / "random_forest.joblib",
)
SCALER_PATH = first_existing(
    REPO_ROOT / "artifacts/v2_16khz/features/scaler.json",
    LOCAL_ARTIFACT_DIR / "scaler.json",
)
PUBLIC_PROFESSOR_BUNDLE = REPO_ROOT / "artifacts/v2_16khz/features/public_professor_features.npz"
PRIVATE_REAL_BUNDLE = REPO_ROOT / "artifacts/v2_16khz/features/private_real_features.npz"
PUBLIC_REFERENCE = REPO_ROOT / "artifacts/v2_16khz/evaluation/public_professor/random_forest.json"
PRIVATE_REFERENCE = REPO_ROOT / "artifacts/v2_16khz/evaluation/real_only/random_forest.json"
PRIVATE_AUDIO_ROOT = REPO_ROOT / "data/raw/private"

RUN_PUBLIC_BUNDLE_EVALUATION = PUBLIC_PROFESSOR_BUNDLE.exists()
RUN_PRIVATE_BUNDLE_EVALUATION = PRIVATE_REAL_BUNDLE.exists()
RUN_RAW_PRIVATE_REBUILD = PRIVATE_AUDIO_ROOT.exists()
WRITE_OUTPUT_FILES = False
OUT_DIR = REPO_ROOT / "notebooks/random_forest_16khz_results"

print(f"repo_root:                         {REPO_ROOT}")
print(f"model:                             {MODEL_PATH} (exists={MODEL_PATH.exists()})")
print(f"scaler:                            {SCALER_PATH} (exists={SCALER_PATH.exists()})")
print(f"run_public_bundle_evaluation:      {RUN_PUBLIC_BUNDLE_EVALUATION}")
print(f"run_private_bundle_evaluation:     {RUN_PRIVATE_BUNDLE_EVALUATION}")
print(f"run_raw_private_rebuild:           {RUN_RAW_PRIVATE_REBUILD}")
print(f"write_output_files:                {WRITE_OUTPUT_FILES}")


repo_root:                         /home/kiettqa/drone-audio-classification
model:                             /home/kiettqa/drone-audio-classification/artifacts/v2_16khz/models/v2a/models/random_forest.joblib (exists=True)
scaler:                            /home/kiettqa/drone-audio-classification/artifacts/v2_16khz/features/scaler.json (exists=True)
run_public_bundle_evaluation:      True
run_private_bundle_evaluation:     True
run_raw_private_rebuild:           True
write_output_files:                False


## Steps

### 1. Load and validate frozen artifacts

Các assertion dưới đây từ chối model/scaler sai feature order, sai preprocessing
profile, sai sample rate hoặc khác training bundle fingerprint. Threshold phải là
`threshold_f2` đã được chọn từ validation; notebook không gọi `fit` hay chọn lại
threshold.


In [3]:
_require_file(MODEL_PATH, "Random Forest model")
_require_file(SCALER_PATH, "train-fitted scaler")

scaler = infer._load_scaler(SCALER_PATH)
model = infer._load_model(MODEL_PATH)
infer._validate_model_package("random_forest", model)

assert model["model_type"] == "random_forest"
assert model["threshold_name"] == "threshold_f2"
assert model["preprocessing_profile"] == PREPROCESSING_PROFILE
assert scaler["preprocessing_profile"] == PREPROCESSING_PROFILE
assert int(model["processed_sample_rate_hz"]) == TARGET_SAMPLE_RATE_HZ
assert int(scaler["processed_sample_rate_hz"]) == TARGET_SAMPLE_RATE_HZ
assert model["source_bundle_fingerprint"] == scaler["source_bundle_fingerprint"]
assert tuple(model["feature_names"]) == tuple(scaler["feature_names"]) == infer.FEATURE_NAMES

threshold = float(model["threshold"])
model_size_mb = MODEL_PATH.stat().st_size / (1024 * 1024)
print(f"model_type:             {model['model_type']}")
print(f"model_size:             {model_size_mb:.1f} MiB")
print(f"features:               {len(infer.FEATURE_NAMES)} ({infer.FEATURE_SET})")
print(f"preprocessing_profile:  {model['preprocessing_profile']}")
print(f"processed_sample_rate:  {model['processed_sample_rate_hz']} Hz")
print(f"threshold_name:         {model['threshold_name']}")
print(f"frozen_threshold:       {threshold:.15f}")


model_type:             random_forest
model_size:             14.1 MiB
features:               100 (spectral_mfcc_v1)
preprocessing_profile:  mono_resample_poly_16khz_v1
processed_sample_rate:  16000 Hz
threshold_name:         threshold_f2
frozen_threshold:       0.408520085021445


### 2. Evaluation helpers

Metric definitions match the project evaluator. `PR-AUC` uses the same
rank-by-score accumulation as the production research code, including its tie
behavior, so saved reports can be reconciled exactly.


In [4]:
def safe_div(numerator: float, denominator: float):
    return None if denominator == 0 else numerator / denominator


def fbeta(precision, recall, beta: float):
    if precision is None or recall is None:
        return None
    denominator = (beta * beta * precision) + recall
    return 0.0 if denominator == 0 else (1 + beta * beta) * precision * recall / denominator


def roc_auc_project(labels, scores):
    positive_count = sum(label == POSITIVE_LABEL for label in labels)
    negative_count = sum(label == NEGATIVE_LABEL for label in labels)
    if positive_count == 0 or negative_count == 0:
        return None
    pairs = sorted(zip(scores, labels), key=lambda pair: pair[0])
    rank = 1
    positive_rank_sum = 0.0
    index = 0
    while index < len(pairs):
        score = pairs[index][0]
        end = index + 1
        while end < len(pairs) and pairs[end][0] == score:
            end += 1
        average_rank = (rank + rank + (end - index) - 1) / 2.0
        positive_rank_sum += sum(average_rank for _, label in pairs[index:end] if label == POSITIVE_LABEL)
        rank += end - index
        index = end
    wins = positive_rank_sum - (positive_count * (positive_count + 1) / 2.0)
    return wins / (positive_count * negative_count)


def pr_auc_project(labels, scores):
    positives = sum(label == POSITIVE_LABEL for label in labels)
    if positives == 0:
        return None
    pairs = sorted(zip(scores, labels), reverse=True)
    tp = fp = 0
    previous_recall = area = 0.0
    for _, label in pairs:
        if label == POSITIVE_LABEL:
            tp += 1
        else:
            fp += 1
        recall = tp / positives
        precision = tp / (tp + fp)
        area += (recall - previous_recall) * precision
        previous_recall = recall
    return area


def frozen_metrics(labels, scores, frozen_threshold: float):
    labels = [str(value) for value in labels]
    scores = [float(value) for value in scores]
    predicted = [POSITIVE_LABEL if score >= frozen_threshold else NEGATIVE_LABEL for score in scores]
    tp = sum(y == POSITIVE_LABEL and p == POSITIVE_LABEL for y, p in zip(labels, predicted))
    tn = sum(y == NEGATIVE_LABEL and p == NEGATIVE_LABEL for y, p in zip(labels, predicted))
    fp = sum(y == NEGATIVE_LABEL and p == POSITIVE_LABEL for y, p in zip(labels, predicted))
    fn = sum(y == POSITIVE_LABEL and p == NEGATIVE_LABEL for y, p in zip(labels, predicted))
    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    specificity = safe_div(tn, tn + fp)
    defined = [value for value in (recall, specificity) if value is not None]
    return {
        "rows": len(labels),
        "threshold": frozen_threshold,
        "accuracy": safe_div(tp + tn, len(labels)),
        "balanced_accuracy": sum(defined) / len(defined) if defined else None,
        "precision": precision,
        "recall": recall,
        "f1": fbeta(precision, recall, 1.0),
        "f2": fbeta(precision, recall, 2.0),
        "roc_auc": roc_auc_project(labels, scores),
        "pr_auc": pr_auc_project(labels, scores),
        "false_positive_rate": safe_div(fp, fp + tn),
        "false_negative_rate": safe_div(fn, fn + tp),
        "confusion_matrix": {"tp": tp, "tn": tn, "fp": fp, "fn": fn},
    }


def load_bundle(path: Path):
    _require_file(path, "feature bundle")
    with np.load(path, allow_pickle=False) as data:
        bundle = {
            "features": np.asarray(data["scaled_features"], dtype=np.float64),
            "labels": np.asarray(data["labels"]),
            "splits": np.asarray(data["splits"]),
            "is_professor": np.asarray(data["is_professor_data"], dtype=bool),
            "feature_names": tuple(str(value) for value in data["feature_names"]),
            "feature_set": str(data["feature_set"].item()),
            "preprocessing_profile": str(data["preprocessing_profile"].item()),
            "processed_sample_rate_hz": int(data["processed_sample_rate_hz"].item()),
            "bundle_fingerprint": str(data["bundle_fingerprint"].item()),
            "original_files": np.asarray(data["original_files"]),
            "environments": np.asarray(data["environments"]),
            "window_start_sec": np.asarray(data["window_start_sec"], dtype=np.float64),
            "window_end_sec": np.asarray(data["window_end_sec"], dtype=np.float64),
        }
    assert bundle["feature_set"] == infer.FEATURE_SET
    assert bundle["feature_names"] == infer.FEATURE_NAMES
    assert bundle["preprocessing_profile"] == PREPROCESSING_PROFILE
    assert bundle["processed_sample_rate_hz"] == TARGET_SAMPLE_RATE_HZ
    return bundle


def score_features(features: np.ndarray, label: str):
    return infer._score_model(label, model["estimator"], features, batch_size=8192)


def print_metric_row(name: str, metrics: Mapping[str, Any]):
    cm = metrics["confusion_matrix"]
    def pct(key):
        value = metrics.get(key)
        return "n/a" if value is None else f"{100 * value:.2f}%"
    print(
        f"{name:24s} rows={metrics['rows']:6d} "
        f"precision={pct('precision'):>7s} recall={pct('recall'):>7s} "
        f"F1={pct('f1'):>7s} F2={pct('f2'):>7s} FPR={pct('false_positive_rate'):>7s} "
        f"CM(TN/FP/FN/TP)={cm['tn']}/{cm['fp']}/{cm['fn']}/{cm['tp']}"
    )


def print_professor_recall(metrics: Mapping[str, Any]):
    cm = metrics["confusion_matrix"]
    print(
        f"{'Professor positive-only':24s} rows={metrics['rows']:6d} "
        f"recall={100 * metrics['recall']:.2f}% "
        f"missed_windows={cm['fn']} detected_windows={cm['tp']} "
        "(precision/F1/F2/FPR not interpreted: no negative rows)"
    )


def assert_metrics_match(actual, expected, keys, atol=1e-12):
    assert actual["confusion_matrix"] == expected["confusion_matrix"]
    for key in keys:
        left, right = actual.get(key), expected.get(key)
        if left is None or right is None:
            assert left is right, (key, left, right)
        else:
            assert abs(float(left) - float(right)) <= atol, (key, left, right)


### 3. Fast frozen evaluation from canonical bundles

Đây là đường chạy lại nhanh và chính xác các metric đã báo cáo. Public test gồm
public positive + public negative. Professor positive-only chỉ báo recall.
Private real-only gồm real positive + real negative.


In [5]:
bundle_results = {}
metric_keys = (
    "accuracy", "balanced_accuracy", "precision", "recall", "f1", "f2",
    "roc_auc", "pr_auc", "false_positive_rate", "false_negative_rate",
)

if RUN_PUBLIC_BUNDLE_EVALUATION:
    public_bundle = load_bundle(PUBLIC_PROFESSOR_BUNDLE)
    public_mask = (public_bundle["splits"] == "public_test") & (~public_bundle["is_professor"])
    professor_mask = (public_bundle["splits"] == "professor_test") & public_bundle["is_professor"]

    public_scores = score_features(public_bundle["features"][public_mask], "public_test")
    professor_scores = score_features(public_bundle["features"][professor_mask], "professor_positive_only")
    public_metrics = frozen_metrics(public_bundle["labels"][public_mask], public_scores, threshold)
    professor_metrics = frozen_metrics(public_bundle["labels"][professor_mask], professor_scores, threshold)
    bundle_results["public_test"] = public_metrics
    bundle_results["professor_positive_only"] = professor_metrics
    print_metric_row("Public test", public_metrics)
    print_professor_recall(professor_metrics)

    if PUBLIC_REFERENCE.exists():
        public_reference = json.loads(PUBLIC_REFERENCE.read_text(encoding="utf-8"))
        assert_metrics_match(public_metrics, public_reference["public_test"]["metrics"], metric_keys)
        assert abs(professor_metrics["recall"] - public_reference["professor_test"]["metrics"]["recall"]) <= 1e-12
        print("Public/Professor reference reconciliation: PASS")
else:
    print("Public/Professor bundle replay skipped (public bundle is intentionally optional).")

if RUN_PRIVATE_BUNDLE_EVALUATION:
    private_bundle = load_bundle(PRIVATE_REAL_BUNDLE)
    private_scores = score_features(private_bundle["features"], "private_real_only")
    private_metrics = frozen_metrics(private_bundle["labels"], private_scores, threshold)
    bundle_results["private_real_only"] = private_metrics
    print_metric_row("Private real-only", private_metrics)

    if PRIVATE_REFERENCE.exists():
        private_reference = json.loads(PRIVATE_REFERENCE.read_text(encoding="utf-8"))
        assert_metrics_match(private_metrics, private_reference["metrics"], metric_keys)
        print("Private real-only reference reconciliation: PASS")
else:
    print("Private bundle replay skipped. Set PRIVATE_REAL_BUNDLE and enable its flag.")


score progress: model=public_test windows=8192/67244 rate=129047.5 windows/s elapsed=0.1s eta=0.5s
score progress: model=public_test windows=16384/67244 rate=149386.9 windows/s elapsed=0.1s eta=0.3s
score progress: model=public_test windows=24576/67244 rate=158404.8 windows/s elapsed=0.2s eta=0.3s
score progress: model=public_test windows=32768/67244 rate=166357.9 windows/s elapsed=0.2s eta=0.2s
score progress: model=public_test windows=40960/67244 rate=173721.6 windows/s elapsed=0.2s eta=0.2s
score progress: model=public_test windows=49152/67244 rate=179968.3 windows/s elapsed=0.3s eta=0.1s
score progress: model=public_test windows=57344/67244 rate=176020.1 windows/s elapsed=0.3s eta=0.1s
score progress: model=public_test windows=65536/67244 rate=179595.4 windows/s elapsed=0.4s eta=0.0s
score progress: model=public_test windows=67244/67244 rate=166130.2 windows/s elapsed=0.4s eta=0.0s
score progress: model=professor_positive_only windows=1113/1113 rate=28508.0 windows/s elapsed=0.0s e

### 4. Rebuild Private real-only from raw WAV

Cell này là kiểm tra end-to-end quan trọng nhất: đọc 1 positive recording và 10
negative recordings, resample toàn file về 16 kHz, sau đó mới cắt window và
extract 100 features. Kết quả được so với frozen bundle/reference report.


In [6]:
raw_private_result = None
if RUN_RAW_PRIVATE_REBUILD:
    positive_files = sorted((PRIVATE_AUDIO_ROOT / "professor_real_world").glob("*.wav"))
    negative_files = sorted((PRIVATE_AUDIO_ROOT / "Recordings").glob("*.wav"))
    assert len(positive_files) == 1, f"expected 1 positive recording, found {len(positive_files)}"
    assert len(negative_files) == 10, f"expected 10 negative recordings, found {len(negative_files)}"

    positive_rows, positive_raw, positive_extract = infer._extract_dataset_features(
        positive_files, progress_every=1000
    )
    negative_rows, negative_raw, negative_extract = infer._extract_dataset_features(
        negative_files, progress_every=1000
    )
    raw_rows = positive_rows + negative_rows
    raw_features = np.vstack((positive_raw, negative_raw))
    raw_labels = np.asarray(
        [POSITIVE_LABEL] * len(positive_rows) + [NEGATIVE_LABEL] * len(negative_rows)
    )
    assert len(positive_rows) == 1113
    assert len(negative_rows) == 592
    assert raw_features.shape == (1705, 100)

    scaled_raw = infer._apply_scaler(raw_features, scaler)
    raw_scores = score_features(scaled_raw, "private_real_only_raw_rebuild")
    raw_private_result = frozen_metrics(raw_labels, raw_scores, threshold)
    print_metric_row("Private raw rebuild", raw_private_result)
    print(
        "raw cohort: "
        f"positive_recordings={len(positive_files)}, positive_windows={len(positive_rows)}, "
        f"negative_recordings={len(negative_files)}, negative_windows={len(negative_rows)}"
    )
    print(
        "non-finite replacements: "
        f"positive={positive_extract['non_finite_replacements']}, "
        f"negative={negative_extract['non_finite_replacements']}"
    )

    if "private_real_only" in bundle_results:
        assert_metrics_match(raw_private_result, bundle_results["private_real_only"], metric_keys)
        print("Raw-WAV vs canonical bundle reconciliation: PASS")

    if WRITE_OUTPUT_FILES:
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        detections = np.asarray(raw_scores) >= threshold
        infer._write_window_csv(OUT_DIR / "private_real_only_windows.csv", raw_rows, raw_scores, detections)
        payload = {
            "model": str(MODEL_PATH),
            "preprocessing_profile": PREPROCESSING_PROFILE,
            "processed_sample_rate_hz": TARGET_SAMPLE_RATE_HZ,
            "threshold_name": model["threshold_name"],
            "threshold": threshold,
            "counts": {"drone": len(positive_rows), "no_drone": len(negative_rows)},
            "metrics": raw_private_result,
        }
        (OUT_DIR / "private_real_only_summary.json").write_text(
            json.dumps(infer._jsonable(payload), indent=2, sort_keys=True) + "\n",
            encoding="utf-8",
        )
        print(f"Wrote new outputs under: {OUT_DIR}")
else:
    print("Skipped raw private rebuild. Set PRIVATE_AUDIO_ROOT and RUN_RAW_PRIVATE_REBUILD=True.")


feature progress: windows=1000 files=1/1 rate=570.0 windows/s elapsed=1.8s current_file=SMU08662_20260608_135349.wav
feature progress: file=1/1 windows=1113 source_rate=16000 processed_rate=16000 path=SMU08662_20260608_135349.wav
feature progress: windows=1113 files=1/1 rate=578.5 windows/s elapsed=1.9s current_file=SMU08662_20260608_135349.wav
feature progress: file=1/10 windows=135 source_rate=48000 processed_rate=16000 path=recording_20260715_124928_traffic.wav
feature progress: file=2/10 windows=121 source_rate=48000 processed_rate=16000 path=recording_20260715_125131_traffic.wav
feature progress: file=3/10 windows=20 source_rate=48000 processed_rate=16000 path=recording_20260715_125247_traffic.wav
feature progress: file=4/10 windows=20 source_rate=48000 processed_rate=16000 path=recording_20260715_125301_traffic.wav
feature progress: file=5/10 windows=20 source_rate=48000 processed_rate=16000 path=recording_20260715_125318_traffic.wav
feature progress: file=6/10 windows=10 source_

## Checks

Expected frozen RF results:

| Dataset | Precision | Recall | F1 | F2 | FPR |
| --- | ---: | ---: | ---: | ---: | ---: |
| Public test | 98.43% | 99.63% | 99.03% | 99.39% | 0.83% |
| Professor positive-only | n/a | 82.48% | n/a | n/a | n/a |
| Private real-only | 94.74% | 82.48% | 88.18% | 84.67% | 8.61% |

Private real-only confusion matrix phải là `TN=541, FP=51, FN=195, TP=918`.
Nếu assertion fail, không nên dùng kết quả: thường là do sai model/scaler/profile,
khác feature order, resample sau windowing, hoặc threshold bị thay đổi.


## Optional: inference on another WAV folder

Sửa `INFERENCE_AUDIO_ROOT` thành một thư mục WAV. Vì folder tùy ý không có label,
cell chỉ xuất detection rate và score distribution, không tính F1/precision/recall.


In [7]:
INFERENCE_AUDIO_ROOT = None  # Example: Path("/path/to/new/wav_folder")
MAX_INFERENCE_FILES = 0      # 0 = all files

if INFERENCE_AUDIO_ROOT is not None:
    inference_root = Path(INFERENCE_AUDIO_ROOT)
    _require_dir(inference_root, "inference audio root")
    inference_files = infer._discover_wavs(inference_root, include_calibration=True)
    if MAX_INFERENCE_FILES:
        inference_files = inference_files[:MAX_INFERENCE_FILES]
    if not inference_files:
        raise ValueError(f"no WAV files found below {inference_root}")
    inference_rows, inference_raw, inference_extract = infer._extract_dataset_features(
        inference_files, progress_every=1000
    )
    inference_scaled = infer._apply_scaler(inference_raw, scaler)
    inference_scores = score_features(inference_scaled, "unlabeled_folder")
    inference_detections = inference_scores >= threshold
    inference_summary = infer._summarize_scores(
        inference_rows, inference_scores, inference_detections, threshold
    )
    print(json.dumps(infer._jsonable(inference_summary), indent=2, sort_keys=True))
else:
    print("Optional unlabeled folder inference is disabled.")


Optional unlabeled folder inference is disabled.


## Next steps

- Notebook này dùng cho **frozen inference/evaluation**, không dùng để retrain.
- Không thay threshold dựa trên Professor/private real-only.
- Khi có positive recordings độc lập mới, append thành external test mới và báo
  cả recording-level uncertainty; không gộp chúng vào train/validation.
- Để ghi report CSV/JSON mới, bật `WRITE_OUTPUT_FILES=True`; artifact cũ không bị
  overwrite vì output đi vào namespace riêng của notebook.
